# FeatureUnion — Revision Notes

## 1. What is FeatureUnion?

`FeatureUnion` is a scikit-learn utility used to apply **multiple transformers in parallel** to the same input and then **concatenate their outputs**.

### Mental Model

```text
                    ┌── Transformer 1 ──┐
                    │                   │
X ──────────────────┼── Transformer 2 ──┼──→ Concatenate → Final X
                    │                   │
                    └── Transformer 3 ──┘

### Example:

from sklearn.pipeline import FeatureUnion

union = FeatureUnion([
    ('scale', StandardScaler()),
    ('pca', PCA(n_components=2))
])

X_new = union.fit_transform(X)

Each transformer receives X, transforms it independently, and the resulting features are concatenated.

2. Why use FeatureUnion?

Use FeatureUnion when you want to obtain different representations/features from the same input and combine them.

For example:

X
├── Transformer A → Features A ──┐
├── Transformer B → Features B ──┼──→ Combined Features
└── Transformer C → Features C ──┘
3. Important: What does "whole dataset" mean?

This is an important source of confusion.

When we say:

"FeatureUnion applies transformers to the whole dataset."

It means:

By default, each transformer branch receives the entire input X passed to the FeatureUnion.

It does NOT mean:

"FeatureUnion cannot work with selected columns."

For example:

FeatureUnion([
    ('transformer_1', transformer_1),
    ('transformer_2', transformer_2)
])

Both transformers receive the same X:

                    ┌── transformer_1(X)
X ──────────────────┤
                    └── transformer_2(X)

FeatureUnion itself does not automatically divide X into columns.

4. Can FeatureUnion work on selected columns?

Yes.

You can explicitly select columns before/during the branches using appropriate column-selection transformers.

For example, conceptually:

                 ┌── Select f1 → Transformer A ──┐
X ───────────────┤                               ├──→ Concatenate
                 └── Select f2,f3 → Transformer B┘

So the important distinction is:

FeatureUnion = parallel processing

It does not mean:

FeatureUnion = automatically split columns.

5. FeatureUnion vs Pipeline
Pipeline

Pipeline performs transformations sequentially.

X
 ↓
Scaler
 ↓
PCA
 ↓
Output
Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2))
])

The output of one step becomes the input of the next step.

FeatureUnion

FeatureUnion performs transformations in parallel.

              ┌── Scaler ──┐
X ────────────┤             ├──→ Concatenate
              └── PCA ──────┘

The transformers independently process the input and their outputs are combined.

6. FeatureUnion vs ColumnTransformer

This distinction is important for interviews.

Tool	Main purpose
Pipeline	Sequential transformations
FeatureUnion	Parallel transformations + concatenate outputs
ColumnTransformer	Apply different transformations to different columns
Example

Suppose:

f1 = Age
f2 = Salary
f3 = City

If we want:

Age + Salary → StandardScaler
City         → OneHotEncoder

ColumnTransformer is the natural choice:

ColumnTransformer([
    ('num', StandardScaler(), ['Age', 'Salary']),
    ('cat', OneHotEncoder(), ['City'])
])
7. Can they be nested?

Yes.

Pipeline inside ColumnTransformer

Very common:

ColumnTransformer([
    ('num',
     Pipeline([
         ('imputer', SimpleImputer()),
         ('scaler', StandardScaler())
     ]),
     numerical_columns)
])

Here:

Selected columns
      ↓
  Imputation
      ↓
  Scaling
FeatureUnion inside ColumnTransformer

Also possible, but less common.

Useful when the same selected columns need multiple parallel transformations:

Selected columns
       │
       ├── Transformer A ──┐
       └── Transformer B ──┴──→ Concatenate
8. Interview Mental Model

Remember these three words:

ColumnTransformer → WHERE?
Pipeline          → AFTER WHAT?
FeatureUnion      → ALONGSIDE WHAT?

Or:

ColumnTransformer → Select columns
Pipeline          → Sequential
FeatureUnion      → Parallel
9. Common Interview Trap
❌ Incorrect

"FeatureUnion splits the dataset into different columns and applies different transformations."

✅ Correct

"FeatureUnion applies multiple transformers in parallel to the same input by default and concatenates their outputs. If different columns need different transformations, column selection must be handled explicitly; ColumnTransformer is usually the more natural tool."

10. One-line Definition

FeatureUnion combines the outputs of multiple transformers that operate in parallel on the input features.

Key takeaway
Pipeline     = A → B → C
FeatureUnion = A ┐
               B ├→ concatenate
               C ┘
ColumnTransformer = different columns → different transformations

In [6]:
import pandas as pd
import numpy as np
from sklearn.pipeline import FeatureUnion
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [7]:
# Generating a random dataset with 10 rows and 4 columns
np.random.seed(42)  # For reproducibility
data = np.random.randn(10, 4)

# Creating a DataFrame and naming the columns
df = pd.DataFrame(data, columns=['f1', 'f2', 'f3', 'y'])

In [10]:
df

,f1,f2,f3,y
0,0.496714,-0.138264,0.647689,1.523030
1,-0.234153,-0.234137,1.579213,0.767435
2,-0.469474,0.542560,-0.463418,-0.465730
3,0.241962,-1.913280,-1.724918,-0.562288
4,-1.012831,0.314247,-0.908024,-1.412304
5,1.465649,-0.225776,0.067528,-1.424748
6,-0.544383,0.110923,-1.150994,0.375698
7,-0.600639,-0.291694,-0.601707,1.852278
8,-0.013497,-1.057711,0.822545,-1.220844
9,0.208864,-1.959670,-1.328186,0.196861


In [8]:
# Define FeatureUnion
feature_union = FeatureUnion([
    ('scaler', StandardScaler()),  # Apply StandardScaler
    ('pca', PCA(n_components=2))   # Apply PCA, reduce to 2 components
])

In [9]:
feature_union

,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('scaler', ...), ('pca', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",2
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False


In [13]:
X = df.iloc[:,:3]

In [14]:
X

,f1,f2,f3
0,0.496714,-0.138264,0.647689
1,-0.234153,-0.234137,1.579213
2,-0.469474,0.542560,-0.463418
3,0.241962,-1.913280,-1.724918
4,-1.012831,0.314247,-0.908024
5,1.465649,-0.225776,0.067528
6,-0.544383,0.110923,-1.150994
7,-0.600639,-0.291694,-0.601707
8,-0.013497,-1.057711,0.822545
9,0.208864,-1.959670,-1.328186


In [16]:
X_transformed = feature_union.fit_transform(X)

In [17]:
X_transformed

array([[ 0.81529269,  0.41836017,  0.94787822,  1.0256589 , -0.42541278],
       [-0.28229178,  0.30277673,  1.87370087,  1.77253158, -0.3582229 ],
       [-0.63568645,  1.23915754, -0.15642721,  0.32788806,  1.03874199],
       [ 0.43271761, -1.72158742, -1.41020602, -1.91107241, -0.68996004],
       [-1.45167552,  0.96390523, -0.59831227, -0.1931527 ,  1.37166183],
       [ 2.27039577,  0.31285628,  0.3712689 ,  0.51175952, -0.89113254],
       [-0.74818047,  0.71877826, -0.83979461, -0.48428039,  1.02073088],
       [-0.83266319,  0.23338663, -0.2938697 , -0.19172257,  0.58395797],
       [ 0.04907985, -0.69011866,  1.12166434,  0.72687801, -0.8114612 ],
       [ 0.38301148, -1.77751476, -1.01590253, -1.58448799, -0.83890321]])

In [19]:
feature_union.get_feature_names_out()

array(['scaler__f1', 'scaler__f2', 'scaler__f3', 'pca__pca0', 'pca__pca1'],
      dtype=object)

In [21]:
new_df = pd.DataFrame(X_transformed , columns = feature_union.get_feature_names_out())

In [22]:
new_df

,scaler__f1,scaler__f2,scaler__f3,pca__pca0,pca__pca1
0,0.815293,0.418360,0.947878,1.025659,-0.425413
1,-0.282292,0.302777,1.873701,1.772532,-0.358223
2,-0.635686,1.239158,-0.156427,0.327888,1.038742
3,0.432718,-1.721587,-1.410206,-1.911072,-0.689960
4,-1.451676,0.963905,-0.598312,-0.193153,1.371662
5,2.270396,0.312856,0.371269,0.511760,-0.891133
6,-0.748180,0.718778,-0.839795,-0.484280,1.020731
7,-0.832663,0.233387,-0.293870,-0.191723,0.583958
8,0.049080,-0.690119,1.121664,0.726878,-0.811461
9,0.383011,-1.777515,-1.015903,-1.584488,-0.838903
